# Clasificación
El objetivo de este notebook es probar si es posible, a partir de dev_set_clean.csv, clasificar si una propiedad pertenece a las normales o a los outliers.

Esto nos permitiría usar un modelo entrenado específicamente para ese tipo de propiedad para predecir su precio.

In [7]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn import set_config
from scipy.stats import randint, uniform

from src.config import TREE_BASED_CONFIG_NORMAL
from src.pipeline import build_feature_pipeline
from src.utils import DEV_SET_CLEAN_PATH, COLS
from src.model_evaluation import classifier_search_cv
set_config(transform_output="pandas")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Creamos nuevo target is_outlier que es:
- 1 si la propiedad se corresponde con el 1.5% más caro
- 0 si la propiedad se corresponde con el 98.5% más barato

In [8]:
df_full = pd.read_csv("../" + DEV_SET_CLEAN_PATH)
threshold = df_full[COLS.TARGET].quantile(0.985)
df_full['is_outlier'] = (df_full[COLS.TARGET] >= threshold).astype(int)

class_dist = df_full['is_outlier'].value_counts(normalize=True).sort_index()
print(f"  Normales (0):  {class_dist[0]:.1%} ({(df_full['is_outlier']==0).sum():,})")
print(f"  Outliers (1):  {class_dist[1]:.1%} ({(df_full['is_outlier']==1).sum():,})")

# Features y target
X_clf = df_full.drop(columns=[COLS.TARGET, 'is_outlier'])
y_clf = df_full['is_outlier']

  Normales (0):  98.5% (266,659)
  Outliers (1):  1.5% (4,063)


Creamos un dataset balanceado con undersampling:

In [9]:
scale_pos_weight = (y_clf == 0).sum() / (y_clf == 1).sum()
print(f"\n⚖️  Scale pos weight: {scale_pos_weight:.1f}")
print(f"   (Compensar desbalance 98.5% vs 1.5%)")


⚖️  Scale pos weight: 65.6
   (Compensar desbalance 98.5% vs 1.5%)


## 1. Dataset Desbalanceado
Hacemos random search para encontrar los mejores hiperparámetros

In [10]:
clf_param_grid = {
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.19),  # [0.01, 0.20]
    'n_estimators': [200, 300, 500, 700, 800, 900],
    'min_child_weight': randint(1, 30),
    'subsample': uniform(0.5, 0.5),  # [0.5, 1.0]
    'colsample_bytree': uniform(0.5, 0.5),  # [0.5, 1.0]
    'gamma': uniform(0.0, 0.5),
    'reg_alpha': uniform(0.0, 3.0),
    'reg_lambda': uniform(0.0, 5.0),
    'scale_pos_weight': [scale_pos_weight * 0.5,
                         scale_pos_weight,
                         scale_pos_weight * 1.5,
                         scale_pos_weight * 2.0],
}

In [11]:
clf_results_full = classifier_search_cv(
    model_class=XGBClassifier,
    param_grid=clf_param_grid,
    X=X_clf,
    y=y_clf,
    feature_pipeline=build_feature_pipeline(TREE_BASED_CONFIG_NORMAL),
    n_iter=600,
    n_splits=3,
    scoring_metric='f1',  # Optimizar F1 (balance precision/recall)
    verbose=True
)


🔍 RandomizedSearchCV - CLASIFICACIÓN
   Combinaciones: 600 × 3 folds = 1800 fits
   Modelo: XGBClassifier
   Métrica objetivo: f1
   Proporción clases: 266659:4063



Optimizando:   0%|          | 0/1800 [00:00<?, ?it/s]

/Users/gonza/Documents/Gonza/UdeSA/05-machine-learning/alquileres/venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



✅ MEJOR CONFIGURACIÓN (por f1):
   Test Precision: 0.792 (79.2%)
   Test Recall:    0.770 (77.0%)
   Test F1-Score:  0.781
   Test ROC-AUC:   0.915

   Train vs Test (overfitting check):
   Precision gap: -16.9%
   Recall gap:    -23.0%
   F1 gap:        -20.0%

   Mejores hiperparámetros:
     colsample_bytree: 0.7792021248679025
     gamma: 0.21211100462348814
     learning_rate: 0.18220733316799984
     max_depth: 9.0
     min_child_weight: 3.0
     n_estimators: 800.0
     reg_alpha: 0.034060934302257206
     reg_lambda: 2.3433032099706312
     scale_pos_weight: 65.63106079251784
     subsample: 0.8704869304553342


In [12]:
clf_results_full.to_csv(f'../results/random_search/clf_full.csv')

## 2. Dataset balanceado

In [13]:
df_normal = df_full[df_full['is_outlier'] == 0]
df_outliers = df_full[df_full['is_outlier'] == 1]

# Undersample clase normal a exactamente 4063
df_normal_sampled = df_normal.sample(n=40000, random_state=42)
# Combinar
df_balanced = pd.concat([df_normal_sampled, df_outliers], ignore_index=True)
# Shuffle para mezclar
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_clf_balanced = df_balanced.drop(columns=[COLS.TARGET, 'is_outlier'])
y_clf_balanced = df_balanced['is_outlier']

In [14]:
clf_results_balanced = classifier_search_cv(
    model_class=XGBClassifier,
    param_grid=clf_param_grid,
    X=X_clf_balanced,
    y=y_clf_balanced,
    feature_pipeline=build_feature_pipeline(TREE_BASED_CONFIG_NORMAL),
    n_iter=800,  # 100 iteraciones
    n_splits=5,
    scoring_metric='f1',
    verbose=True
)



🔍 RandomizedSearchCV - CLASIFICACIÓN
   Combinaciones: 800 × 5 folds = 4000 fits
   Modelo: XGBClassifier
   Métrica objetivo: f1
   Proporción clases: 40000:4063



Optimizando:   0%|          | 0/4000 [00:00<?, ?it/s]


✅ MEJOR CONFIGURACIÓN (por f1):
   Test Precision: 0.844 (84.4%)
   Test Recall:    0.831 (83.1%)
   Test F1-Score:  0.838
   Test ROC-AUC:   0.931

   Train vs Test (overfitting check):
   Precision gap: -11.4%
   Recall gap:    -16.9%
   F1 gap:        -14.2%

   Mejores hiperparámetros:
     colsample_bytree: 0.5758374398663756
     gamma: 0.15586103389777411
     learning_rate: 0.0572129365647485
     max_depth: 9.0
     min_child_weight: 1.0
     n_estimators: 900.0
     reg_alpha: 0.6136337795083898
     reg_lambda: 3.570320411118071
     scale_pos_weight: 32.81553039625892
     subsample: 0.9383828183808747


In [15]:
clf_results_full.to_csv(f'../results/random_search/clf_balanced.csv')